# **Chess outcome prediction**

### **Imports**
These libraries are used throughout this jupyter notebook. 
Check the README.md to see how to install all of the required libraries and dependencies. 

In [ ]:
import sklearn as sk
import pandas as pd
import numpy as np
import chess.pgn
import matplotlib.pyplot as plt
import seaborn
from sklearn.pipeline import (Pipeline, make_pipeline)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


## **Loading data from the parquet files**

In [355]:
df_raw = pd.read_parquet("parsed")

## **Data preprocessing**

In [377]:
df = df_raw.copy()
df["Event"] = df["Event"].astype(str)

df = df[(df["Event"]=="Rated Bullet game") | (df["Event"]=="Rated Blitz game")]

for column in ["WhiteElo", "BlackElo"]:
    df[column] = pd.to_numeric(df[column], errors="coerce").astype("Int64")
df = df[(df["WhiteElo"] <= 1500) & (df["BlackElo"] <= 1500)]

df["ResultRaw"] = df["ResultRaw"].astype(str)
df["ECO"] = df["ECO"].astype(str)
df["Opening"] = df["Opening"].astype(str)

df["Datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], format="%Y.%m.%d %H:%M:%S")

df["TimeControl"] = df["TimeControl"].astype(str)
df["Termination"] = df["Termination"].astype(str)

df = df[~df["TimeControl"].str.startswith("0+")].copy() 

df = df.drop(columns=["Date", "Time", "Termination"])

## **Feature engineering**

### **Utility functions**
These are functions used on the dataframe in order extract features from the data.

In [357]:
def to_winprob(s):
    return s.white().wdl(model="lichess").expectation()

def transform_result(str_res):
    if str_res == "1-0":
        return 1.0
    elif str_res == "0-1":
        return -1.0
    else:
        return 0.0

def early_castle(san_list):
    if len(san_list) > 20:
        checkpoint = 20
    else:
        checkpoint = len(san_list)

    white = False
    black = False
    for i in range(0, checkpoint):
        if "O-O" in san_list[i]:
            if i % 2 == 0:
                white = True
            else:
                black = True

    return white, black

def early_queen(san_list):
    if len(san_list) > 20:
        checkpoint = 20
    else:
        checkpoint = len(san_list)

    white = False
    black = False
    for i in range(0, checkpoint):
        if "Q" in san_list[i]:
            if i % 2 == 0:
                white = True
            else:
                black = True

    return white, black

# def early_advantage(win_prob):
    if all(x is None for x in win_prob[:20]):
        return None

    if len(win_prob) > 20:
        checkpoint = 20
    else:
        checkpoint = len(win_prob)

    early_advantage = False
    for i in range(0, checkpoint):
        if win_prob[i] is not None and win_prob[i] > 0.55:
            early_advantage = True

    return early_advantage


def early_wp_mean(win_prob):
    values = []
    for x in win_prob[:20]:
        values.append(x)
    
    return sum(values) / len(values)

def early_double_move(san_list):
    if len(san_list) > 10:
        checkpoint = 10
    else:
        checkpoint = len(san_list)
    
    board = chess.Board()
    moved_white = set()
    moved_black = set()
    white = False
    black = False

    for i in range(0, checkpoint):
        move = board.push_san(san_list[i])

        if i % 2 == 0:
            if move.from_square in moved_white:
                white = True
            moved_white.discard(move.from_square)
            moved_white.add(move.to_square)
            moved_black.discard(move.to_square)
        else:
            if move.from_square in moved_black:
                black = True
            moved_black.discard(move.from_square)
            moved_black.add(move.to_square)
            moved_white.discard(move.to_square)

    return white, black

def parse_tc(tc):
    base, inc = tc.split("+")
    return int(base), int(inc)

def clock_features(row):
    base, inc = parse_tc(row["TimeControl"])
    clocks = row["Clocks"]
    if len(clocks) < 20 or clocks[18] is None or clocks[19] is None:
        return None, None, None

    white_left = clocks[18]
    black_left = clocks[19]

    white_used = (base + 10 * inc - white_left) / base
    black_used = (base + 10 * inc - black_left) / base

    diffs = []
    for i in range(0, 20, 2):
        white = clocks[i]
        black = clocks[i + 1]
        diffs.append((white - black) / base)
    avg_clock_diff = sum(diffs) / len(diffs) if diffs else None

    return white_used, black_used, avg_clock_diff

VALUES = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9}

def early_material_advantage(sans):
    board = chess.Board()
    for san in sans[:20]:
        board.push_san(san)
    return sum(v * (len(board.pieces(p, chess.WHITE)) - len(board.pieces(p, chess.BLACK)))
               for p, v in VALUES.items())

### **Feature extraction**

In [378]:
parts = df["Opening"].str.split(":", n=1)
df["OpeningFamily"] = parts.str[0].str.strip()
df["OpeningVariation"] = parts.str[1].str.strip() 

df["Result"] = df["ResultRaw"].apply(transform_result)
df["EarlyWPMean"] = df["WinProb"].apply(early_wp_mean)
df["EloDiff"] = df["WhiteElo"]-df["BlackElo"]
df["AvgElo"] = (df["WhiteElo"] + df["BlackElo"]) / 2


df[["WhiteEarlyCastle", "BlackEarlyCastle"]] = pd.DataFrame(df["Sans"].apply(early_castle).tolist(), index=df.index)
df[["WhiteEarlyQueen", "BlackEarlyQueen"]] = pd.DataFrame(df["Sans"].apply(early_queen).tolist(), index=df.index)
df[["WhiteEarlyDoubleMove", "BlackEarlyDoubleMove"]] = pd.DataFrame(df["Sans"].apply(early_double_move).tolist(), index=df.index)
df[["WhiteTimeUsed", "BlackTimeUsed", "AvgClockDiff"]] = pd.DataFrame(df.apply(clock_features, axis=1).tolist(), index=df.index)

top = df["OpeningFamily"].value_counts().head(20).index
df["OpeningGroup"] = df["OpeningFamily"].where(df["OpeningFamily"].isin(top), "Other")

df[["BaseTime", "Increment"]] = df["TimeControl"].str.split("+", expand=True).astype(int)
df["EstDuration"] = df["BaseTime"] + 40 * df["Increment"] # the Lichess formula for differentiating blitz from bullet games.
df["EarlyMaterialAdvantage"] = df["Sans"].apply(early_material_advantage)

df = df.drop(["Opening", "ResultRaw", "OpeningVariation", "WhiteElo", "BlackElo", "OpeningFamily", "OpeningVariation", "ECO", "BaseTime", "Increment", "TimeControl", "Event", "Clocks", "Sans", "WinProb", "Datetime"], axis=1)

df = pd.get_dummies(df, columns=["OpeningGroup"], drop_first=True)


## **Data visualisation**

## **Logistic regression**

In [380]:
num_cols = ["EloDiff", "AvgElo", "WhiteTimeUsed", "BlackTimeUsed", "AvgClockDiff", "EstDuration"]
bin_cols = [c for c in df.columns if c.startswith(("White", "Black", "OpeningGroup_"))
            and c not in num_cols]

d = df[df["Result"] != 0] # by getting rid of draws, I'm making it a binary classification problem 
X = d[num_cols + bin_cols]
X[num_cols] = X[num_cols].astype(float)
y = (d["Result"] == 1).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)
X_train = pipeline.fit_transform(X_train)
X_test = pipeline.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
print("test accuracy:", model.score(X_test, y_test))

test accuracy: 0.5661881977671451


This is not that good of a result generally speaking. It means that the features I chose coupled with LogisticRegression explain only ~8% of the variation in the data. For the problem I chose, it is hard for traditional ML to capture a lot of the variation in the data. I need to get more features from the data and experiment with different models, maybe also rethink the core idea of my analysis. I will also look more deeply into what I can do with the chess python library and Stockfish evaluations.